# BƯỚC 4: HỆ THỐNG GỢI Ý VIỆC LÀM DỰA TRÊN NỘI DUNG (CONTENT-BASED FILTERING ENGINE)
## DỰ ÁN: IT JOB MARKET INTELLIGENCE
---
**Phần chính** (trình bày theo thứ tự):
1. Nạp dữ liệu sạch & **lọc tin thuộc ngành IT**
2. Chuẩn hóa kỹ năng & xây dựng **Content Soup**
3. Vector hóa văn bản **TF-IDF** (tokenizer giữ nguyên C++/C#/.NET và tiếng Việt có dấu)
4. **Chấm điểm đa thành phần**: 5 điểm thành phần, co giãn trọng số, lời giải thích, báo cáo 1:1
5. Thử nghiệm thực tế (case studies)
6. **Đánh giá**: Precision@5, NDCG@5, đánh giá không dùng tiêu đề, **bảng kết quả chính**

**Phụ lục** (bằng chứng cho các lựa chọn thiết kế, mở khi cần): A. Khảo sát dữ liệu · B. Phân bố điểm · C. Quét trọng số kỹ năng : văn bản

## TÓM TẮT

| | |
|---|---|
| **Bài toán** | Người dùng nhập hồ sơ (vị trí, kỹ năng, kinh nghiệm, nơi làm, lương mong muốn, hoặc dán CV) ➔ xếp hạng các tin IT **còn hạn** phù hợp nhất, kèm **lời giải thích** |
| **Dữ liệu** | `jobs_clean.csv` (notebook 02), chỉ giữ tin `is_it = True` |
| **Biểu diễn tin** | *Content Soup* = Tiêu đề ×3 + Kỹ năng ×4 + Thẻ ngành ×1 + Mô tả ➔ vector **TF-IDF** |
| **Biểu diễn hồ sơ** | *Profile Soup* = Vị trí ×3 + Kỹ năng ×4 + CV ➔ cùng không gian TF-IDF |
| **Công thức điểm** | $0{,}30\,S_{\text{kỹ năng}} + 0{,}40\,S_{\text{văn bản}} + 0{,}10\,S_{\text{kinh nghiệm}} + 0{,}10\,S_{\text{vị trí}} + 0{,}10\,S_{\text{lương}}$; |
| **Điểm kỹ năng** | độ phủ × $\frac{m}{m+1}$, với $m$ = số kỹ năng khớp + 0,5 × số kỹ năng *gần nhau* (PostgreSQL ≈ MySQL) |
| **Điểm văn bản** | cosine TF-IDF, chia cho cosine lớn nhất (tin giống nhất đạt 1,0) |
| **Đánh giá** | 25 hồ sơ mẫu × 5 nhóm nghề, top-5; đo thêm khi **bỏ tiêu đề khỏi truy vấn** để tránh "trùng chữ" với nhãn |
| **Kết quả** | xem **bảng kết quả chính ở mục 6.2** (tự cập nhật khi chạy lại với dữ liệu mới) |

## 1. Nạp Tập Dữ Liệu Sạch (Cleaned Dataset Ingestion)

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Tự động xác định đường dẫn gốc dự án
current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
project_root = current_dir
while not os.path.exists(os.path.join(project_root, 'data')) and os.path.dirname(project_root) != project_root:
    project_root = os.path.dirname(project_root)

csv_path = os.path.join(project_root, 'data', 'notebook', 'content_base', 'jobs_clean.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join(project_root, 'data', 'cleaned', 'jobs_clean.csv')

# 2. Đọc file CSV dữ liệu sạch mới nhất
jobs_all = pd.read_csv(csv_path)
for col in ['required_skills', 'job_tags', 'description_clean']:
    jobs_all[col] = jobs_all[col].fillna('')
assert {'is_it', 'job_tags'} <= set(jobs_all.columns), "Thiếu cột is_it/job_tags: hãy chạy lại notebook 02 trước."
print(f" Đã nạp dữ liệu sạch: {jobs_all.shape[0]:,} dòng x {jobs_all.shape[1]} cột  ({csv_path})")

# 3. Chỉ giữ tin thuộc ngành IT. Giữ lại jobs_all để so sánh ở phần đánh giá (ablation)
jobs = jobs_all[jobs_all['is_it']].reset_index(drop=True)
print(f" Sau khi lọc ngành IT: {len(jobs):,} tin ({len(jobs) / len(jobs_all):.1%}) | loại {len(jobs_all) - len(jobs):,} tin ngoài ngành")
jobs.drop(columns=['job_url', 'company_logo_url']).head(3)   # ẩn 2 cột link (chỉ app dùng) cho dễ nhìn

 Đã nạp dữ liệu sạch: 5,234 dòng x 19 cột  (c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\notebook\content_base\jobs_clean.csv)
 Sau khi lọc ngành IT: 3,675 tin (70.2%) | loại 1,559 tin ngoài ngành


,job_id,company_name,title,description_clean,required_skills,job_tags,experience_min,experience_max,location,city,salary_min,salary_max,employment_type,work_mode,deadline,is_active,is_it
0,topcv_2314643,ZIGExN VeNtura,Full-Stack Web Engineer Intern/Fresher (AI Cod...,COMPANY INTRODUCTION\nAbout our parent company...,"Python, Java, JavaScript, C#, PHP, Golang, Rub...","Fullstack Developer, IT - Phần mềm, Tiếng Anh ...",0.0,0.0,"Floor 11, Vina Building, 131 Xo Viet Nghe Tinh...",Hồ Chí Minh,NaN,NaN,Intern,Onsite,2026-10-28,False,True
1,topcv_2314547,Điện tử R&P,Kỹ Sư Hệ Thống Nhúng,"R&P là công ty công nghệ, hoạt động trên lĩnh ...","C++, Linux, ERP, Embedded","Embedded Engineer/Lập trình nhúng, Điện / Điện...",0.0,0.0,"65 Lê Trung Nghĩa, Phường Bảy Hiền, Hồ Chí Minh",Hồ Chí Minh,NaN,NaN,Full-time,Onsite,2026-10-28,False,True
2,topcv_2321616,FOXSCORE,UI/UX Designer - Mobile App,FOXSCORE – NỀN TẢNG THỂ THAO THÔNG MINH\nRa đờ...,"Android, iOS, Figma, UX Design","Product Owner/Product Manager, IT - Phần mềm, ...",2.0,2.0,"62/6-8-10 Trương Công Định, Phường Tân Bình, H...",Hồ Chí Minh,15000000.0,15000000.0,Full-time,Onsite,2026-11-05,False,True


## 2. Chuẩn Hóa Kỹ Năng & Xây Dựng "Content Soup" (Feature Engineering)

### 1. Kỹ năng công nghệ vs. thẻ ngành

- `required_skills`: **chỉ kỹ năng công nghệ** (`Python`, `SQL`, `Docker`...), dùng cho điểm khớp kỹ năng;
- `job_tags`: thẻ ngành/lĩnh vực/chức danh (`IT - Phần mềm`, `Ngân hàng`, `QA Engineer`...), chỉ dùng làm ngữ cảnh văn bản.

Từ điển kỹ năng dưới đây dùng để **bóc kỹ năng từ đoạn CV** do người dùng nhập, và chuẩn hóa kỹ năng người dùng chọn (`ReactJS` ➔ `React`).

### 2. Xây dựng "Content Soup" có gán trọng số
$$\text{Content Soup} = (\text{Title} \times 3) + (\text{Required Skills} \times 4) + (\text{Job Tags} \times 1) + (\text{Description} \times 1)$$



In [2]:
import sys
if project_root not in sys.path:          
    sys.path.insert(0, project_root)

# Từ điển kỹ năng: configs/dictionaries/skills.yaml | hàm chuẩn hóa/bóc kỹ năng: src/data_cleaning/skill_canonical.py (dùng CHUNG với notebook 02 và ứng dụng)
from src.data_cleaning.skill_canonical import (
    CANONICAL_SKILL_MAP, TECH_EXTRA, TECH_LOOKUP, extract_skills_from_text, normalize_user_skills,
)
print(f"Từ điển kỹ năng: {len(TECH_LOOKUP):,} cách viết ➔ {len(set(TECH_LOOKUP.values())):,} kỹ năng chuẩn")
print("Chuẩn hóa kỹ năng người dùng nhập:", normalize_user_skills(['reactjs', 'k8s', 'postgres', 'Python']))

# Xây dựng Content Soup gán trọng số
def build_content_soup(row):
    title = str(row.get('title', '')) if pd.notnull(row.get('title')) else ''
    skills = str(row.get('required_skills', '')) if pd.notnull(row.get('required_skills')) else ''
    tags = str(row.get('job_tags', '')) if pd.notnull(row.get('job_tags')) else ''
    desc = str(row.get('description_clean', '')) if pd.notnull(row.get('description_clean')) else ''
    return f"{title} {title} {title} {skills} {skills} {skills} {skills} {tags} {desc}".strip()

jobs['content_soup'] = jobs.apply(build_content_soup, axis=1)

print("=== KIỂM TRA BÓC TÁCH KỸ NĂNG TỪ ĐOẠN CV ===")
print(extract_skills_from_text("Có 2 năm làm C++, C#, .NET Core, CI/CD, Node.js. Gửi CV về HR. Go to market."))
print("\n=== MẪU CONTENT SOUP CỦA 2 CÔNG VIỆC ĐẦU TIÊN ===")
for idx, s in enumerate(jobs['content_soup'].head(2)):
    print(f"\n[Job {idx+1}] {jobs.iloc[idx]['title']}:")
    print(s[:250] + "...")

Từ điển kỹ năng: 268 cách viết ➔ 174 kỹ năng chuẩn
Chuẩn hóa kỹ năng người dùng nhập: ['React', 'Kubernetes', 'PostgreSQL', 'Python']
=== KIỂM TRA BÓC TÁCH KỸ NĂNG TỪ ĐOẠN CV ===
['C++', 'C#', 'Node.js', '.NET', 'CI/CD']

=== MẪU CONTENT SOUP CỦA 2 CÔNG VIỆC ĐẦU TIÊN ===

[Job 1] Full-Stack Web Engineer Intern/Fresher (AI Coding: PHP / Ruby / Python):
Full-Stack Web Engineer Intern/Fresher (AI Coding: PHP / Ruby / Python) Full-Stack Web Engineer Intern/Fresher (AI Coding: PHP / Ruby / Python) Full-Stack Web Engineer Intern/Fresher (AI Coding: PHP / Ruby / Python) Python, Java, JavaScript, C#, PHP,...

[Job 2] Kỹ Sư Hệ Thống Nhúng:
Kỹ Sư Hệ Thống Nhúng Kỹ Sư Hệ Thống Nhúng Kỹ Sư Hệ Thống Nhúng C++, Linux, ERP, Embedded C++, Linux, ERP, Embedded C++, Linux, ERP, Embedded C++, Linux, ERP, Embedded Embedded Engineer/Lập trình nhúng, Điện / Điện tử / Điện lạnh, Công nghệ kỹ thuật R...


## 3. Vector Hóa Văn Bản (TF-IDF)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

# 1. Chuẩn hóa token IT đặc thù TRƯỚC khi tách từ (regex áp dụng trên chuỗi đã lower-case)
_TECH_REPLACEMENTS = [
    (re.compile(r"(?<![\w+])c\+\+(?![\w+])"), " cpp "),
    (re.compile(r"(?<![\w#])c#(?![\w#])"), " csharp "),
    (re.compile(r"(?<![\w])(?:asp)?\.net(?:\s*core)?(?![\w])"), " dotnet "),
    (re.compile(r"(?<![\w])node\.?js(?![\w])"), " nodejs "),
    (re.compile(r"(?<![\w])react\.?js(?![\w])"), " reactjs "),
    (re.compile(r"(?<![\w])vue\.?js(?![\w])"), " vuejs "),
    (re.compile(r"(?<![\w])next\.?js(?![\w])"), " nextjs "),
    (re.compile(r"(?<![\w])ci\s*/\s*cd(?![\w])"), " cicd "),
    (re.compile(r"(?<![\w])a\s*/\s*b\s+test"), " abtest"),
    (re.compile(r"(?<![\w])pl\s*/\s*sql(?![\w])"), " plsql "),
    (re.compile(r"(?<![\w])ui\s*/\s*ux(?![\w])|(?<![\w])ux\s*/\s*ui(?![\w])"), " uiux "),
    (re.compile(r"(?<![\w])front[\s-]?end(?![\w])"), " frontend "),
    (re.compile(r"(?<![\w])back[\s-]?end(?![\w])"), " backend "),
    (re.compile(r"(?<![\w])full[\s-]?stack(?![\w])"), " fullstack "),
]

def normalize_tech_text(text):
    s = str(text).lower()
    for pat, rep in _TECH_REPLACEMENTS:
        s = pat.sub(rep, s)
    return s

TOKEN_PATTERN = r"(?u)\b\w\w+\b"      # chữ/số Unicode (giữ tiếng Việt có dấu), tối thiểu 2 ký tự

# 2. Ví dụ: tokenizer giữ nguyên C++, C#, .NET, CI/CD, Node.js và chữ tiếng Việt có dấu
sample = "C++ C# .NET CI/CD Node.js kinh nghiệm lập trình"
tokens = TfidfVectorizer(preprocessor=normalize_tech_text, token_pattern=TOKEN_PATTERN).build_analyzer()(sample)
print(f"Văn bản: {sample}")
print(f"Token  : {tokens}\n")

# 3. Huấn luyện TF-IDF trên Content Soup
start_time = time.time()
tfidf = TfidfVectorizer(
    preprocessor=normalize_tech_text,
    token_pattern=TOKEN_PATTERN,
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf.fit_transform(jobs['content_soup'])

elapsed = time.time() - start_time
print(f" Đã vector hóa thành công: {tfidf_matrix.shape[0]:,} jobs x {tfidf_matrix.shape[1]:,} features")
print(f" Thời gian thực thi TF-IDF: {elapsed:.2f} giây")

Văn bản: C++ C# .NET CI/CD Node.js kinh nghiệm lập trình
Token  : ['cpp', 'csharp', 'dotnet', 'cicd', 'nodejs', 'kinh', 'nghiệm', 'lập', 'trình']

 Đã vector hóa thành công: 3,675 jobs x 10,000 features
 Thời gian thực thi TF-IDF: 6.89 giây


## 4. Thiết Kế Hệ Thống Chấm Điểm Đa Thành Phần (Composite Scoring Engine)

### Công thức điểm tổng hợp:
$$\text{Score}(j, u) = 0.30 S_{skills} + 0.40 S_{text} + 0.10 S_{experience} + 0.10 S_{location/work\ mode} + 0.10 S_{salary}$$

### 4.1. Các hàm tính điểm thành phần chuẩn hóa [0, 1] & Cơ chế co giãn trọng số

In [4]:
# Nhóm kỹ năng THAY THẾ ĐƯỢC cho nhau: configs/dictionaries/skill_groups.yaml | hàm chia khớp / gần / thiếu: src/data_cleaning/skill_canonical.py
from src.data_cleaning.skill_canonical import SKILL_GROUPS, RELATED_SKILLS, split_skill_match

RELATED_SKILL_CREDIT = 0.5   # một kỹ năng gần nhau = nửa kỹ năng khớp (chọn ở mục 6.1)
print(f"{len(SKILL_GROUPS)} nhóm kỹ năng gần nhau, ví dụ:")
for grp in [SKILL_GROUPS[2], SKILL_GROUPS[14], SKILL_GROUPS[17]]:
    print("  •", ", ".join(grp))

def compute_skill_score(user_skills, job_required_skills):
    """
    S_skills [0, 1] = độ phủ × m/(m+1), với m = số kỹ năng khớp chính xác + RELATED_SKILL_CREDIT × số kỹ năng gần nhau.
    Hệ số m/(m+1): khớp càng nhiều kỹ năng thì bằng chứng càng chắc (tin chỉ yêu cầu 1 kỹ năng không đạt ngay 100%).
    Trả về None nếu tin không ghi kỹ năng nào: thành phần này bị bỏ và trọng số còn lại được co giãn.
    """
    req_set = {s.lower() for s in job_required_skills if s and s.lower() != 'nan'} if job_required_skills else set()
    if not req_set:
        return None
    matched, related, _ = split_skill_match(user_skills, list(req_set))
    m = len(matched) + RELATED_SKILL_CREDIT * len(related)
    coverage = m / len(req_set)
    return round(coverage * m / (m + 1), 3)

print("Ví dụ: hồ sơ PostgreSQL + Vue, job yêu cầu MySQL + React + Docker")
print("  split_skill_match ➔", split_skill_match(["PostgreSQL", "Vue"], ["MySQL", "React", "Docker"]))
print("  điểm kỹ năng =", compute_skill_score(["PostgreSQL", "Vue"], ["MySQL", "React", "Docker"]))

def compute_exp_score(user_exp, job_min_exp, job_max_exp=None):
    """S_experience: Điểm phù hợp số năm kinh nghiệm."""
    if pd.isna(job_min_exp) or job_min_exp == 0:
        return 1.0
    diff = user_exp - job_min_exp
    if diff >= 0:
        if pd.notna(job_max_exp) and user_exp > job_max_exp + 3:
            return 0.85 # Hơi overqualified
        return 1.0
    elif diff >= -1.0:
        return 0.7 # Thiếu dưới 1 năm
    elif diff >= -2.0:
        return 0.4 # Thiếu 1-2 năm
    else:
        return 0.1 # Thiếu quá nhiều

def compute_location_mode_score(user_city, user_modes, job_city, job_mode):
    """S_location/mode: Điểm kết hợp địa điểm và mô hình làm việc."""
    score_mode = 0.5
    score_city = 0.5
    job_mode_str = str(job_mode).lower()
    
    if isinstance(user_modes, str):
        user_modes = [user_modes]
    user_modes_lower = [m.lower() for m in user_modes] if user_modes else ['tất cả']
    
    if 'tất cả' in user_modes_lower or not user_modes:
        score_mode = 1.0
    elif any(m in job_mode_str for m in user_modes_lower):
        score_mode = 1.0
    elif 'remote' in job_mode_str:
        score_mode = 0.9 # Job Remote phù hợp gần như mọi ứng viên
    elif 'hybrid' in job_mode_str and any('remote' in m for m in user_modes_lower):
        score_mode = 0.8
    else:
        score_mode = 0.2
        
    if user_city == 'Tất cả' or not user_city:
        score_city = 1.0
    elif str(job_city).lower() == str(user_city).lower():
        score_city = 1.0
    elif 'remote' in job_mode_str:
        score_city = 1.0 # Tin Remote không rào cản địa lý
    else:
        score_city = 0.1
        
    return round(0.5 * score_city + 0.5 * score_mode, 3)

def compute_salary_score(user_min_salary, job_salary_min, job_salary_max):
    """S_salary: Trả về None nếu bài đăng không có số liệu lương cụ thể."""
    if pd.isna(job_salary_max) and pd.isna(job_salary_min):
        return None
    max_sal = job_salary_max if pd.notna(job_salary_max) else job_salary_min
    if not user_min_salary or user_min_salary == 0:
        return 1.0
    if max_sal >= user_min_salary:
        return 1.0
    else:
        ratio = max_sal / user_min_salary
        return round(max(0.0, float(ratio)), 3)

def compute_composite_score(s_skills, s_text, s_exp, s_loc, s_sal, base_weights=None):
    if base_weights is None:
        base_weights = {'skills': 0.30, 'text': 0.40, 'exp': 0.10, 'loc': 0.10, 'sal': 0.10}
    scores = {'skills': s_skills, 'text': s_text, 'exp': s_exp, 'loc': s_loc, 'sal': s_sal}
    available = {k: w for k, w in base_weights.items() if scores[k] is not None}
    total_w = sum(available.values())
    weights = {k: w / total_w for k, w in available.items()}
    total_score = sum(weights[k] * scores[k] for k in weights)
    return round(total_score, 4), weights

37 nhóm kỹ năng gần nhau, ví dụ:
  • React, Vue, Angular, Svelte
  • SQL, PostgreSQL, MySQL, SQL Server, Oracle Database, MariaDB, SQLite, T-SQL, PL/SQL
  • AWS, Azure, GCP, Cloud
Ví dụ: hồ sơ PostgreSQL + Vue, job yêu cầu MySQL + React + Docker
  split_skill_match ➔ ([], ['MySQL', 'React'], ['Docker'])
  điểm kỹ năng = 0.167


### 4.2. Các hàm hỗ trợ hiển thị & Sinh lời giải thích (Rationale Generator)

In [5]:
def generate_rationale(matched_skills, missing_skills, user_exp, job_min_exp, job_mode, is_salary_fit,
                       related_skills=None):
    """Sinh câu giải thích gợi ý trực quan."""
    reasons = []
    related_skills = related_skills or []
    total_skills = len(matched_skills) + len(related_skills) + len(missing_skills)
    if total_skills > 0:
        matched_str = ', '.join(matched_skills[:3]) if matched_skills else 'Chưa trùng trực tiếp'
        reasons.append(f"Khớp {len(matched_skills)}/{total_skills} kỹ năng ({matched_str})")
        if related_skills:
            reasons.append(f"Gần với {', '.join(related_skills[:3])}")
    if pd.isna(job_min_exp) or job_min_exp == 0:
        reasons.append("Không đòi hỏi số năm kinh nghiệm")
    elif user_exp >= job_min_exp:
        reasons.append(f"Kinh nghiệm phù hợp ({user_exp:.0f} năm >= yêu cầu {job_min_exp:.0f} năm)")
    else:
        reasons.append(f"Yêu cầu {job_min_exp:.0f} năm (bạn có {user_exp:.0f} năm)")
    reasons.append(f"Hình thức {job_mode}")
    if is_salary_fit is True:
        reasons.append("Đạt kỳ vọng lương")
    elif is_salary_fit is None:
        reasons.append("Lương thỏa thuận")
    else:
        reasons.append("Thấp hơn kỳ vọng lương")
    return "; ".join(reasons)

def format_salary_display(s_min, s_max):
    """Định dạng hiển thị mức lương trực quan."""
    if pd.notnull(s_min) and pd.notnull(s_max):
        if s_min == s_max:
            return f"{s_min/1e6:.0f} tr"
        return f"{s_min/1e6:.0f} - {s_max/1e6:.0f} tr"
    elif pd.notnull(s_max):
        return f"Lên đến {s_max/1e6:.0f} tr"
    elif pd.notnull(s_min):
        return f"Từ {s_min/1e6:.0f} tr"
    return "Thoả thuận"

# ============================================================================
# CHẠY THỬ 1: lời giải thích cho MỘT HỒ SƠ MẪU trên 3 tin thật có đặc điểm khác nhau
# ============================================================================
demo_skills = ['Python', 'SQL', 'PostgreSQL', 'Docker', 'React']
demo_exp, demo_min_salary = 2, 20_000_000
print(f"Hồ sơ mẫu: kỹ năng {', '.join(demo_skills)} | {demo_exp} năm kinh nghiệm | lương mong muốn ≥ {demo_min_salary / 1e6:.0f} triệu\n")

def _req(x):
    return [s.strip() for s in str(x).split(',') if s.strip() and s.strip().lower() != 'nan']

def _explain(row):
    req = _req(row['required_skills'])
    matched, related, missing = split_skill_match(demo_skills, req)
    s_sal = compute_salary_score(demo_min_salary, row['salary_min'], row['salary_max'])
    sal_fit = None if s_sal is None else bool(s_sal == 1.0)
    return req, matched, related, missing, generate_rationale(
        matched, missing, demo_exp, row['experience_min'], row['work_mode'], sal_fit, related)

# Chọn 3 tin minh họa 3 tình huống: khớp tốt + đủ lương | thiếu kinh nghiệm + có kỹ năng gần nhau | lương thỏa thuận
info = jobs.apply(lambda r: pd.Series(_explain(r), index=['req', 'matched', 'related', 'missing', 'why']), axis=1)
n_m, n_r = info['matched'].map(len), info['related'].map(len)
has_sal = jobs['salary_min'].notna() | jobs['salary_max'].notna()
sal_ok = jobs[['salary_min', 'salary_max']].max(axis=1) >= demo_min_salary
cases = {
    'Khớp tốt, đủ lương': (n_m >= 3) & sal_ok & (jobs['experience_min'].fillna(0) <= demo_exp),
    'Thiếu kinh nghiệm, có kỹ năng gần nhau': (n_m >= 1) & (n_r >= 1) & (jobs['experience_min'] > demo_exp),
    'Lương thỏa thuận': (n_m >= 2) & ~has_sal,
}
for name, mask in cases.items():
    if not mask.any():
        print(f"(Không có tin nào cho tình huống: {name})\n")
        continue
    i = mask.idxmax()
    r, x = jobs.loc[i], info.loc[i]
    print(f"━━ {name}: {r['title']} — {r['company_name']} ({r['city']})")
    print(f"   Kỹ năng job yêu cầu : {', '.join(x['req'])}")
    print(f"   Khớp                : {', '.join(x['matched']) or '—'}")
    print(f"   Gần với (tính 0,5)  : {', '.join(x['related']) or '—'}")
    print(f"   Còn thiếu           : {', '.join(x['missing']) or '—'}")
    exp_txt = f"{r['experience_min']:g} năm" if pd.notna(r['experience_min']) else 'không ghi'
    print(f"   Kinh nghiệm yêu cầu : {exp_txt} | Hình thức: {r['work_mode']} | "
          f"Lương: {format_salary_display(r['salary_min'], r['salary_max'])}")
    print(f"   ➔ LỜI GIẢI THÍCH    : {x['why']}\n")

# ============================================================================
# CHẠY THỬ 2: format_salary_display trên các dạng lương có thật trong dữ liệu
# ============================================================================
smin, smax = jobs['salary_min'], jobs['salary_max']
salary_cases = {
    'Một mức cố định (min = max)': smin.notna() & (smin == smax),
    'Khoảng lương (min < max)': smin.notna() & smax.notna() & (smin < smax),
    'Chỉ có mức trần ("Tới ...")': smin.isna() & smax.notna(),
    'Chỉ có mức sàn ("Từ ...")': smin.notna() & smax.isna(),
    'Không có số (thỏa thuận)': smin.isna() & smax.isna(),
}
rows = []
for name, mask in salary_cases.items():
    if mask.any():
        r = jobs.loc[mask.idxmax()]
        rows.append({'Dạng lương': name, 'Số tin': int(mask.sum()), 'salary_min (VNĐ)': r['salary_min'],
                     'salary_max (VNĐ)': r['salary_max'], 'Hiển thị': format_salary_display(r['salary_min'], r['salary_max'])})
display(pd.DataFrame(rows).style.format({'salary_min (VNĐ)': '{:,.0f}', 'salary_max (VNĐ)': '{:,.0f}', 'Số tin': '{:,}'}, na_rep='—'))

Hồ sơ mẫu: kỹ năng Python, SQL, PostgreSQL, Docker, React | 2 năm kinh nghiệm | lương mong muốn ≥ 20 triệu

━━ Khớp tốt, đủ lương: Fullstack Developer — HIPPO TECHNOLOGY COMPANY LIMITED (Hồ Chí Minh)
   Kỹ năng job yêu cầu : Python, JavaScript, TypeScript, Django, FastAPI, Node.js, React, Vue, PostgreSQL, MySQL, Git, Agile
   Khớp                : Python, React, PostgreSQL
   Gần với (tính 0,5)  : Django, FastAPI, Vue, MySQL
   Còn thiếu           : JavaScript, TypeScript, Node.js, Git, Agile
   Kinh nghiệm yêu cầu : 1 năm | Hình thức: Onsite | Lương: 23 tr
   ➔ LỜI GIẢI THÍCH    : Khớp 3/12 kỹ năng (Python, React, PostgreSQL); Gần với Django, FastAPI, Vue; Kinh nghiệm phù hợp (2 năm >= yêu cầu 1 năm); Hình thức Onsite; Đạt kỳ vọng lương

━━ Thiếu kinh nghiệm, có kỹ năng gần nhau: Fullstack Developer (C#) — TUẦN CHÂU (Hà Nội)
   Kỹ năng job yêu cầu : JavaScript, TypeScript, C#, SQL, .NET, REST API, Microservices, React, HTML/CSS, PostgreSQL, SQL Server, Redis, Docker, CI/CD, Git, Linux

,Dạng lương,Số tin,salary_min (VNĐ),salary_max (VNĐ),Hiển thị
0,Một mức cố định (min = max),565,"15,000,000","15,000,000",15 tr
1,Khoảng lương (min < max),723,"8,000,000","12,000,000",8 - 12 tr
2,"Chỉ có mức trần (""Tới ..."")",264,—,"16,000,000",Lên đến 16 tr
3,"Chỉ có mức sàn (""Từ ..."")",30,"25,300,000",—,Từ 25 tr
4,Không có số (thỏa thuận),"2,093",—,—,Thoả thuận


### 4.3. Bộ chấm điểm dùng chung (`score_components` ➔ `combine_scores`) và hàm gợi ý `recommend_jobs`

Để **hàm gợi ý và báo cáo 1:1 luôn cho cùng một con số**, toàn bộ phép tính được tách thành 3 bước dùng chung:

1. `score_components(...)`: tính 5 điểm thành phần cho **mọi** job (mảng, `NaN` = thiếu dữ liệu);
2. `combine_scores(...)`: gộp bằng trọng số, tự co giãn lại khi có thành phần `NaN`;
3. `recommend_jobs(...)` / `analyze_cv_fit_for_job(...)`: chỉ việc sắp xếp / trình bày kết quả.

$$\text{Score}(u, j) = \frac{\sum_{k \in A_j} w_k \cdot S_k(u, j)}{\sum_{k \in A_j} w_k}, \qquad A_j = \{\text{thành phần job } j \text{ có dữ liệu}\}$$

In [6]:
DEFAULT_WEIGHTS = {'skills': 0.30, 'text': 0.40, 'exp': 0.10, 'loc': 0.10, 'sal': 0.10}
COMPONENTS = ['skills', 'text', 'exp', 'loc', 'sal']


def score_components(target_title="", cv_text="", skills=None, user_exp=1.0, preferred_city="Tất cả",
                     preferred_work_modes=None, min_salary=None):
    """
    Bước 1: tính 5 điểm thành phần cho TẤT CẢ job.
    Trả về dict gồm: user_skills, raw_sim (cosine thô), comps (DataFrame 5 cột, NaN = thiếu dữ liệu), active (mask tin còn hạn).
    """
    all_user_skills = list(dict.fromkeys(normalize_user_skills(skills) + (extract_skills_from_text(cv_text) if cv_text else [])))

    parts = []
    if target_title:
        parts.extend([target_title] * 3)
    if all_user_skills:
        parts.extend([" ".join(all_user_skills)] * 4)
    if cv_text:
        parts.append(cv_text)
    profile_soup = " ".join(parts).strip()
    if not profile_soup:
        return None

    raw_sim = cosine_similarity(tfidf.transform([profile_soup]), tfidf_matrix).flatten()
    # Chuẩn hóa theo truy vấn: job giống nhất đạt 1.0 (cosine thô hiếm khi vượt 0.5 nên sẽ bị các thành phần khác lấn át)
    text_scores = raw_sim / raw_sim.max() if raw_sim.max() > 0 else raw_sim

    # Tách sẵn danh sách kỹ năng của từng job một lần (tăng tốc khi gọi nhiều lần)
    if 'req_list' not in jobs.columns:
        jobs['req_list'] = jobs['required_skills'].map(
            lambda x: [s.strip() for s in str(x).split(',') if s.strip() and s.strip().lower() != 'nan'])

    rows = []
    for idx, row in enumerate(jobs.itertuples(index=False)):
        s_skill = compute_skill_score(all_user_skills, row.req_list)
        s_exp = compute_exp_score(user_exp, row.experience_min, row.experience_max)
        s_loc = compute_location_mode_score(preferred_city, preferred_work_modes, row.city, row.work_mode)
        s_sal = compute_salary_score(min_salary, row.salary_min, row.salary_max)
        rows.append((np.nan if s_skill is None else s_skill, float(text_scores[idx]), s_exp, s_loc,
                     np.nan if s_sal is None else s_sal))
    comps = pd.DataFrame(rows, columns=COMPONENTS)
    return {'user_skills': all_user_skills, 'raw_sim': raw_sim, 'comps': comps, 'active': jobs['is_active'].values.astype(bool)}


def combine_scores(comps, weights=None):
    """
    Bước 2: gộp 5 điểm thành phần thành tổng điểm [0, 1] cho mọi job.
    Thành phần NaN bị loại và trọng số còn lại được co giãn về tổng 1.0 (cùng quy tắc với compute_composite_score).
    Job không còn thành phần nào có trọng số (vd. chỉ chấm kỹ năng mà job không ghi kỹ năng) được 0 điểm.
    """
    w = np.array([(weights or DEFAULT_WEIGHTS)[k] for k in COMPONENTS], dtype=float)
    M = comps[COMPONENTS].values
    available_weight = (~np.isnan(M) * w).sum(axis=1)
    weighted = np.nansum(M * w, axis=1)
    return np.divide(weighted, available_weight, out=np.zeros_like(weighted), where=available_weight > 0)


def recommend_jobs(
    target_title="", cv_text="", skills=None, user_exp=1.0, preferred_city="Tất cả",
    preferred_work_modes=None, min_salary=None, top_n=10, weights=None
):
    """Gợi ý việc làm: Vị trí + Kỹ năng + Văn bản CV + Kinh nghiệm + Địa điểm/Hình thức + Lương mong muốn."""
    sc = score_components(target_title, cv_text, skills, user_exp, preferred_city, preferred_work_modes,
                          min_salary)
    if sc is None:
        print(" Vui lòng cung cấp ít nhất một thông tin: Vị trí (target_title), Kỹ năng (skills), hoặc Văn bản CV (cv_text).")
        return pd.DataFrame()

    total = combine_scores(sc['comps'], weights)
    total = np.where(sc['active'], total, -1.0)                 # bỏ tin hết hạn
    top_idx = np.argsort(-total, kind='stable')[:top_n]

    out = []
    for idx in top_idx:                                          # chỉ sinh lời giải thích cho top_n
        row, c = jobs.iloc[idx], sc['comps'].iloc[idx]
        matched, related, missing = split_skill_match(sc['user_skills'], row['req_list'])
        sal_fit = None if np.isnan(c['sal']) else bool(c['sal'] == 1.0)
        out.append({
            'job_id': row['job_id'], 'Tiêu đề': row['title'], 'Công ty': row['company_name'],
            'Tổng điểm (%)': round(total[idx] * 100, 1),
            'Điểm kỹ năng (%)': None if np.isnan(c['skills']) else round(c['skills'] * 100, 1),
            'Điểm văn bản (%)': round(c['text'] * 100, 1),
            'Cosine gốc (%)': round(float(sc['raw_sim'][idx]) * 100, 1),
            'Điểm kinh nghiệm (%)': round(c['exp'] * 100, 1),
            'Điểm vị trí/hình thức (%)': round(c['loc'] * 100, 1),
            'Lời giải thích': generate_rationale(matched, missing, user_exp, row['experience_min'], row['work_mode'], sal_fit, related),
            'Kinh nghiệm yêu cầu': f"{row['experience_min']} năm" if pd.notnull(row['experience_min']) else 'Không yêu cầu',
            'Hình thức': row['work_mode'], 'Thành phố': row['city'],
            'Mức lương': format_salary_display(row['salary_min'], row['salary_max']),
            'job_url': row['job_url'], 'company_logo_url': row['company_logo_url']})
    return pd.DataFrame(out)

# ============================================================================
# CHẠY THỬ: đi qua 3 bước với hồ sơ mẫu của mục 4.2
# ============================================================================
demo_profile = dict(target_title="Backend Developer", skills=demo_skills, user_exp=demo_exp,
                    preferred_city="Hà Nội", min_salary=demo_min_salary)
COMP_NAME = {'skills': 'Kỹ năng', 'text': 'Văn bản', 'exp': 'Kinh nghiệm', 'loc': 'Vị trí/hình thức', 'sal': 'Lương'}
print("Hồ sơ mẫu:", {k: v for k, v in demo_profile.items()})

# ---- BƯỚC 1: score_components ➔ 5 điểm thành phần cho MỌI tin
sc_demo = score_components(**demo_profile)
total_demo = combine_scores(sc_demo['comps'])
print(f"\nBƯỚC 1 · score_components: ma trận {sc_demo['comps'].shape[0]:,} tin × 5 điểm thành phần")
print("   Tỷ lệ tin THIẾU dữ liệu (NaN) ở từng thành phần:",
      ' | '.join(f"{COMP_NAME[k]} {v:.0%}" for k, v in sc_demo['comps'].isna().mean().items()))

# ---- BƯỚC 2: combine_scores ➔ tính tay cho 2 tin để thấy cơ chế co giãn trọng số
order = np.argsort(-np.where(sc_demo['active'], total_demo, -1.0), kind='stable')
i_full = next(i for i in order if not sc_demo['comps'].iloc[i].isna().any())
i_miss = next(i for i in order if np.isnan(sc_demo['comps'].iloc[i]['sal']) and not np.isnan(sc_demo['comps'].iloc[i]['skills']))

def weight_breakdown(i):
    c = sc_demo['comps'].iloc[i]
    w = pd.Series(DEFAULT_WEIGHTS)
    w_eff = (w[c.notna()] / w[c.notna()].sum()).reindex(COMPONENTS)      # co giãn trên các thành phần có dữ liệu
    t = pd.DataFrame({'Điểm thành phần': c, 'Trọng số gốc': w, 'Trọng số sau co giãn': w_eff,
                      'Đóng góp = điểm × trọng số': c * w_eff})
    t.loc['TỔNG'] = [np.nan, w.sum(), w_eff.sum(), (c * w_eff).sum()]
    return t.rename(index=COMP_NAME)

print("\nBƯỚC 2 · combine_scores: tính tay tổng điểm của 2 tin")
for label, i in [('Tin ĐỦ dữ liệu', i_full), ('Tin THIẾU lương (thỏa thuận) ➔ trọng số lương 0,10 chia lại cho 4 thành phần còn lại', i_miss)]:
    print(f"\n▸ {label}: {jobs.iloc[i]['title']} — {jobs.iloc[i]['company_name']}")
    display(weight_breakdown(i).style.format('{:.1%}', na_rep='— (thiếu, bỏ qua)'))
    manual = weight_breakdown(i).loc['TỔNG', 'Đóng góp = điểm × trọng số']
    print(f"   Tính tay = {manual:.4f} | combine_scores = {total_demo[i]:.4f} ➔ {'KHỚP' if np.isclose(manual, total_demo[i]) else 'LỆCH'}")

# ---- BƯỚC 3: recommend_jobs ➔ sắp xếp, bỏ tin hết hạn, sinh lời giải thích cho top-N
print("\nBƯỚC 3 · recommend_jobs: top-5 tin còn hạn")
rec_demo = recommend_jobs(**demo_profile, top_n=5)
display(rec_demo[['Tiêu đề', 'Công ty', 'Tổng điểm (%)', 'Điểm kỹ năng (%)', 'Điểm văn bản (%)',
                  'Điểm kinh nghiệm (%)', 'Điểm vị trí/hình thức (%)', 'Mức lương']])

Hồ sơ mẫu: {'target_title': 'Backend Developer', 'skills': ['Python', 'SQL', 'PostgreSQL', 'Docker', 'React'], 'user_exp': 2, 'preferred_city': 'Hà Nội', 'min_salary': 20000000}

BƯỚC 1 · score_components: ma trận 3,675 tin × 5 điểm thành phần
   Tỷ lệ tin THIẾU dữ liệu (NaN) ở từng thành phần: Kỹ năng 20% | Văn bản 0% | Kinh nghiệm 0% | Vị trí/hình thức 0% | Lương 57%

BƯỚC 2 · combine_scores: tính tay tổng điểm của 2 tin

▸ Tin ĐỦ dữ liệu: DevOps Engineer (GAME SYSTEM) — KỸ THUẬT CÔNG NGHỆ BLUE DREAM


,Điểm thành phần,Trọng số gốc,Trọng số sau co giãn,Đóng góp = điểm × trọng số
Kỹ năng,22.5%,30.0%,30.0%,6.8%
Văn bản,86.2%,40.0%,40.0%,34.5%
Kinh nghiệm,70.0%,10.0%,10.0%,7.0%
Vị trí/hình thức,55.0%,10.0%,10.0%,5.5%
Lương,100.0%,10.0%,10.0%,10.0%
TỔNG,"— (thiếu, bỏ qua)",100.0%,100.0%,63.7%


   Tính tay = 0.6371 | combine_scores = 0.6371 ➔ KHỚP

▸ Tin THIẾU lương (thỏa thuận) ➔ trọng số lương 0,10 chia lại cho 4 thành phần còn lại: Full Stack Developer (Python/Reactjs) — MAYCONNECT


,Điểm thành phần,Trọng số gốc,Trọng số sau co giãn,Đóng góp = điểm × trọng số
Kỹ năng,40.9%,30.0%,33.3%,13.6%
Văn bản,93.4%,40.0%,44.4%,41.5%
Kinh nghiệm,70.0%,10.0%,11.1%,7.8%
Vị trí/hình thức,100.0%,10.0%,11.1%,11.1%
Lương,"— (thiếu, bỏ qua)",10.0%,"— (thiếu, bỏ qua)","— (thiếu, bỏ qua)"
TỔNG,"— (thiếu, bỏ qua)",100.0%,100.0%,74.1%


   Tính tay = 0.7405 | combine_scores = 0.7405 ➔ KHỚP

BƯỚC 3 · recommend_jobs: top-5 tin còn hạn


,Tiêu đề,Công ty,Tổng điểm (%),Điểm kỹ năng (%),Điểm văn bản (%),Điểm kinh nghiệm (%),Điểm vị trí/hình thức (%),Mức lương
0,Full Stack Developer (Python/Reactjs),MAYCONNECT,74.1,40.9,93.4,70.0,100.0,Thoả thuận
1,DevOps Engineer (GAME SYSTEM),KỸ THUẬT CÔNG NGHỆ BLUE DREAM,63.7,22.5,86.2,70.0,55.0,10 - 20 tr
2,Kỹ Sư DevOps,Công Nghiệp Weldcom,63.4,24.6,72.5,70.0,100.0,25 - 35 tr
3,Chuyên Viên Quản Trị Cơ Sở Dữ Liệu (PostgreSQL),Chứng Khoán SSI,60.3,45.0,59.4,70.0,100.0,Thoả thuận
4,Back-end Developer (trên 3 năm kinh nghiệm),DaouKiwoom Innovation,59.9,24.7,74.9,70.0,55.0,Lên đến 25 tr


### 4.4. Hàm Báo Cáo Đánh Giá Độ Phù Hợp 1:1 (`analyze_cv_fit_for_job`)

Khi người dùng muốn biết **"tôi hợp với tin này đến đâu"**, hàm này in một báo cáo cho **một** tin: tổng điểm, thứ hạng trong toàn bộ tin còn hạn, 5 điểm thành phần, và kỹ năng đã có / gần với / còn thiếu (gợi ý nên học gì thêm).

Hàm dùng chung `score_components` + `combine_scores` với `recommend_jobs`, nên **tổng điểm và thứ hạng phải trùng** với bảng gợi ý. **Chạy thử** bên dưới lấy tin **hạng 3** của mục 4.3, in báo cáo và kiểm tra điều đó.

In [7]:
def analyze_cv_fit_for_job(job_identifier, cv_text="", skills=None, user_exp=1.0, target_title="",
                           preferred_city="Tất cả", preferred_work_modes=None, min_salary=None, weights=None):
    """
    Báo cáo chi tiết độ phù hợp 1:1 giữa hồ sơ và MỘT job. Dùng chung score_components/combine_scores với
    recommend_jobs nên Tổng điểm và thứ hạng luôn khớp với bảng gợi ý khi truyền cùng tham số hồ sơ.
    job_identifier: job_id (khuyên dùng), chỉ số dòng, hoặc một đoạn của tiêu đề.
    """
    if isinstance(job_identifier, (int, np.integer)):
        idx = int(job_identifier)
    else:
        match = jobs[jobs['job_id'] == job_identifier]
        if match.empty:
            match = jobs[jobs['title'].str.contains(str(job_identifier), case=False, na=False, regex=False)]
        if match.empty:
            return f" Không tìm thấy công việc: {job_identifier}"
        idx = match.index[0]

    sc = score_components(target_title, cv_text, skills, user_exp, preferred_city, preferred_work_modes, min_salary)
    if sc is None:
        return " Vui lòng cung cấp ít nhất một thông tin: tiêu đề, kỹ năng hoặc văn bản CV."
    total = combine_scores(sc['comps'], weights)
    rank = int((np.where(sc['active'], total, -1.0) > total[idx]).sum()) + 1

    job, c = jobs.iloc[idx], sc['comps'].iloc[idx]
    req = job['req_list']
    matched, related, missing = split_skill_match(sc['user_skills'], req)
    fmt = lambda v: "không có dữ liệu (bỏ qua)" if np.isnan(v) else f"{v * 100:.1f}%"
    exp_req = job['experience_min'] if pd.notnull(job['experience_min']) else 0.0
    exp_status = (f"Đạt yêu cầu ({user_exp} năm >= {exp_req} năm)" if user_exp >= exp_req
                  else f"Chưa đủ kinh nghiệm ({user_exp} năm < yêu cầu {exp_req} năm)")
    coverage = (f"{len(matched)}/{len(req)} kỹ năng, thêm {len(related)} kỹ năng gần nhau tính 0,5" if req
                else "job không liệt kê kỹ năng")

    return f"""================================================================================
 BÁO CÁO ĐÁNH GIÁ ĐỘ PHÙ HỢP HỒ SƠ VỚI CÔNG VIỆC (1:1 FIT REPORT)
================================================================================
Công việc mục tiêu: {job['title']}
Công ty:             {job['company_name']}
Địa điểm:            {job['city']} ({job['work_mode']})
Mức lương:           {format_salary_display(job['salary_min'], job['salary_max'])}

--- 1. TỔNG ĐIỂM ---
• Tổng điểm: {total[idx] * 100:.1f}%   |   Xếp hạng: {rank}/{int(sc['active'].sum())} tin còn hạn

--- 2. CÁC ĐIỂM THÀNH PHẦN (cùng công thức với recommend_jobs) ---
• Kỹ năng        : {fmt(c['skills'])}   (khớp {coverage})
• Văn bản        : {fmt(c['text'])}   (cosine gốc {sc['raw_sim'][idx] * 100:.1f}%, đã chuẩn hóa theo job giống nhất)
• Kinh nghiệm    : {fmt(c['exp'])}   ({exp_status})
• Vị trí/hình thức: {fmt(c['loc'])}
• Lương          : {fmt(c['sal'])}

--- 3. CHI TIẾT KỸ NĂNG ---
 Kỹ năng bạn đã có ({len(matched)}):
   {', '.join(matched) if matched else 'Chưa trùng trực tiếp kỹ năng công nghệ'}
 Kỹ năng gần với kỹ năng bạn có, học thêm nhanh ({len(related)}):
   {', '.join(related) if related else 'Không có'}
 Kỹ năng còn thiếu mà công ty yêu cầu ({len(missing)}):
   {', '.join(missing) if missing else 'Tuyệt vời! Bạn đáp ứng đầy đủ tất cả kỹ năng'}
================================================================================"""

# ============================================================================
# CHẠY THỬ: báo cáo 1:1 cho tin HẠNG 3 của mục 4.3, rồi kiểm tra khớp với bảng gợi ý
# ============================================================================
pick = rec_demo.iloc[2]
report = analyze_cv_fit_for_job(pick['job_id'], **demo_profile)
print(report)

m = re.search(r"Tổng điểm: ([\d.]+)%\s+\|\s+Xếp hạng: (\d+)/", report)
rep_score, rep_rank = float(m.group(1)), int(m.group(2))
ok = np.isclose(rep_score, pick['Tổng điểm (%)'], atol=0.05) and rep_rank == 3
print(f"\nKIỂM TRA: báo cáo 1:1 ghi {rep_score:.1f}%, hạng {rep_rank} | bảng gợi ý ghi {pick['Tổng điểm (%)']:.1f}%, hạng 3 "
      f"➔ {'KHỚP' if ok else 'LỆCH'}")

 BÁO CÁO ĐÁNH GIÁ ĐỘ PHÙ HỢP HỒ SƠ VỚI CÔNG VIỆC (1:1 FIT REPORT)
Công việc mục tiêu: Kỹ Sư DevOps
Công ty:             Công Nghiệp Weldcom
Địa điểm:            Hà Nội (Onsite)
Mức lương:           25 - 35 tr

--- 1. TỔNG ĐIỂM ---
• Tổng điểm: 63.4%   |   Xếp hạng: 3/1547 tin còn hạn

--- 2. CÁC ĐIỂM THÀNH PHẦN (cùng công thức với recommend_jobs) ---
• Kỹ năng        : 24.6%   (khớp 3/13 kỹ năng, thêm 2 kỹ năng gần nhau tính 0,5)
• Văn bản        : 72.5%   (cosine gốc 19.0%, đã chuẩn hóa theo job giống nhất)
• Kinh nghiệm    : 70.0%   (Chưa đủ kinh nghiệm (2 năm < yêu cầu 3.0 năm))
• Vị trí/hình thức: 100.0%
• Lương          : 100.0%

--- 3. CHI TIẾT KỸ NĂNG ---
 Kỹ năng bạn đã có (3):
   SQL, PostgreSQL, Docker
 Kỹ năng gần với kỹ năng bạn có, học thêm nhanh (2):
   SQL Server, Kubernetes
 Kỹ năng còn thiếu mà công ty yêu cầu (8):
   Cloud, AWS, Azure, GCP, CI/CD, Git, Linux, DevOps

KIỂM TRA: báo cáo 1:1 ghi 63.4%, hạng 3 | bảng gợi ý ghi 63.4%, hạng 3 ➔ KHỚP


## 5. Thử Nghiệm Thực Tế Với Các Kịch Bản Đa Dạng (Case Studies)

Thực nghiệm hệ thống với 4 kịch bản ứng viên:
- **Case 1 (Theo đúng tiêu chuẩn thiết kế mục tiêu)**:
  * Kỹ năng: `Python`, `SQL`, `Machine Learning`
  * Kinh nghiệm: `1.0` năm
  * Địa điểm mong muốn: `Hà Nội`
  * Hình thức làm việc: `Remote` hoặc `Hybrid`
  * Mức lương kỳ vọng: từ `20,000,000` VNĐ (>= 20 triệu)
  * -> Trả về Top 10 kèm Tổng điểm, các điểm thành phần và Lời giải thích trực quan.
- **Case 2**: Ứng viên dán văn bản CV (DevOps/Cloud Microservices tại TP.HCM).
- **Case 3**: Ứng viên Data Analyst kết hợp dán CV và lọc lương tối thiểu.
- **Case 4**: Báo cáo đánh giá chi tiết 1:1 (Fit Report) giữa CV và công việc Top 1.

In [8]:
# --- CASE 1: KỊCH BẢN THIẾT KẾ CHUẨN (Python, SQL, Machine Learning | 1 năm | Hà Nội | Remote/Hybrid | Lương >= 20tr) ---
print("=== CASE 1: ỨNG VIÊN PYTHON, SQL, MACHINE LEARNING (Hà Nội, Remote/Hybrid, Lương >= 20tr) ===")
res1 = recommend_jobs(
    target_title="Python Developer Machine Learning",
    skills=["Python", "SQL", "Machine Learning"],
    user_exp=1.0,
    preferred_city="Hà Nội",
    preferred_work_modes=["Remote", "Hybrid"],
    min_salary=20000000,
    top_n=10
)
display(res1[['Tiêu đề', 'Công ty', 'Tổng điểm (%)', 'Điểm kỹ năng (%)', 'Điểm văn bản (%)', 'Điểm kinh nghiệm (%)', 'Lời giải thích', 'Mức lương']])

# --- CASE 2: ỨNG VIÊN DÁN VĂN BẢN CV (DevOps / Microservices, TP.HCM) ---
print("\n=== CASE 2: ỨNG VIÊN DÁN VĂN BẢN CV (DevOps / Microservices, Hồ Chí Minh) ===")
cv_devops = """
Kỹ sư DevOps với 3 năm kinh nghiệm làm việc với hệ thống Microservices.
Thành thạo Docker, Kubernetes (K8s), xây dựng CI/CD pipeline tự động hóa với GitLab CI và Jenkins.
Kinh nghiệm quản trị hệ thống Linux, triển khai hạ tầng đám mây trên AWS (EC2, S3, RDS, EKS).
"""
res2 = recommend_jobs(
    target_title="DevOps Engineer",
    cv_text=cv_devops,
    user_exp=3.0,
    preferred_city="Hồ Chí Minh",
    top_n=5
)
display(res2[['Tiêu đề', 'Công ty', 'Tổng điểm (%)', 'Điểm kỹ năng (%)', 'Điểm văn bản (%)', 'Lời giải thích', 'Mức lương']])

# --- CASE 3: KẾT HỢP CV + KỸ NĂNG BỔ SUNG + LỌC LƯƠNG TỐI THIỂU >= 15 TRIỆU ---
print("\n=== CASE 3: KẾT HỢP CV + KỸ NĂNG BỔ SUNG + LỌC LƯƠNG >= 15 TRIỆU ===")
cv_da = """
Chuyên viên phân tích dữ liệu với 1 năm kinh nghiệm.
Sử dụng thành thạo SQL để truy vấn và xử lý dữ liệu lớn, thành thạo Excel và Power BI xây dựng Dashboard quản trị kinh doanh.
"""
res3 = recommend_jobs(
    target_title="Data Analyst",
    cv_text=cv_da,
    skills=["Python", "Tableau"],
    user_exp=1.0,
    preferred_city="Tất cả",
    min_salary=15000000,
    top_n=5
)
display(res3[['Tiêu đề', 'Công ty', 'Tổng điểm (%)', 'Điểm kỹ năng (%)', 'Điểm văn bản (%)', 'Lời giải thích', 'Mức lương']])

# --- CASE 4: SO KHỚP CHUYÊN SÂU 1:1 GIỮA CV VÀ CÔNG VIỆC TOP 1 ---
# Truyền ĐÚNG hồ sơ của Case 2 để Tổng điểm trong báo cáo khớp với bảng ở trên (và tra job theo job_id, không theo tiêu đề)
top1_job = res2.iloc[0]['job_id']
rep4 = analyze_cv_fit_for_job(job_identifier=top1_job, cv_text=cv_devops, user_exp=3.0,
                              target_title="DevOps Engineer", preferred_city="Hồ Chí Minh")
print("\n" + rep4)

=== CASE 1: ỨNG VIÊN PYTHON, SQL, MACHINE LEARNING (Hà Nội, Remote/Hybrid, Lương >= 20tr) ===


,Tiêu đề,Công ty,Tổng điểm (%),Điểm kỹ năng (%),Điểm văn bản (%),Điểm kinh nghiệm (%),Lời giải thích,Mức lương
0,AI / Computer Vision Engineer – Fresher/Junior...,DENSO,62.2,45.0,66.7,100.0,"Khớp 2/5 kỹ năng (Python, Machine Learning); G...",11 - 12 tr
1,Data Scientist,Giải pháp Thanh toán Việt Nam (VNPAY),62.2,33.5,74.7,100.0,"Khớp 3/11 kỹ năng (Python, SQL, Machine Learni...",Thoả thuận
2,AI Quantitative Researcher Intern,ĐẦU TƯ FINPROS,55.0,35.7,60.8,100.0,"Khớp 2/5 kỹ năng (Python, Machine Learning); G...",3 - 8 tr
3,Chuyên viên Cao cấp Công nghệ Mô hình - Senior...,NGÂN HÀNG TMCP QUÂN ĐỘI – MBBANK,54.6,33.3,62.9,40.0,"Khớp 2/4 kỹ năng (Python, Machine Learning); Y...",Thoả thuận
4,Junior AI Engineer/ Kỹ Sư AI - Từ 1 Năm Kinh N...,Avepoint,52.7,26.7,58.6,100.0,"Khớp 2/12 kỹ năng (Python, Machine Learning); ...",Thoả thuận
5,Nhân Viên Thiết Kế Xử Lý Ảnh AI,MABUCHI MOTOR,52.3,29.1,55.2,100.0,"Khớp 3/11 kỹ năng (Python, SQL, Machine Learni...",12 - 20 tr
6,Business Analyst (English/Japanese),FPT,50.1,25.0,48.9,70.0,Khớp 1/2 kỹ năng (Machine Learning); Yêu cầu 2...,Lên đến 35 tr
7,AI Specialist,Tập đoàn Công nghệ Thiên Hoàng,49.7,51.7,48.1,40.0,"Khớp 3/9 kỹ năng (Python, SQL, Machine Learnin...",Thoả thuận
8,Kỹ Sư AI/ Máy Học (AI/ML Engineer) - 25-70 Tri...,Thực Phẩm Fresco,48.9,25.0,57.2,70.0,Khớp 1/2 kỹ năng (Machine Learning); Yêu cầu 2...,25 - 70 tr
9,AI Engineer – Multimodal & Agentic RAG,CÔNG NGHỆ UMBALABS,48.9,30.7,58.2,100.0,"Khớp 3/12 kỹ năng (Python, SQL, Machine Learni...",Thoả thuận



=== CASE 2: ỨNG VIÊN DÁN VĂN BẢN CV (DevOps / Microservices, Hồ Chí Minh) ===


,Tiêu đề,Công ty,Tổng điểm (%),Điểm kỹ năng (%),Điểm văn bản (%),Lời giải thích,Mức lương
0,DevOps Engineer,Apero Technologies Group,79.6,54.0,94.7,"Khớp 9/15 kỹ năng (Microservices, AWS, Docker)...",20 - 45 tr
1,System Engineer / DevOps / IT Support,Công nghệ HQSOFT,73.9,57.1,73.4,"Khớp 5/9 kỹ năng (CI/CD, Git, Linux); Gần với ...",Thoả thuận
2,Kỹ Sư DevOps,Công Nghiệp Weldcom,73.7,58.5,76.7,"Khớp 7/13 kỹ năng (AWS, Docker, Kubernetes); G...",25 - 35 tr
3,DevOps Engineer,HOẠT ĐỘNG SỐ TIN CẬY,72.5,55.1,83.0,"Khớp 7/12 kỹ năng (Microservices, Docker, Kube...",Thoả thuận
4,DevOps Engineer,Công Ty Cổ Phần Edulive Toàn Cầu,71.6,50.6,84.5,"Khớp 8/16 kỹ năng (AWS, Docker, Kubernetes); G...",Thoả thuận



=== CASE 3: KẾT HỢP CV + KỸ NĂNG BỔ SUNG + LỌC LƯƠNG >= 15 TRIỆU ===


,Tiêu đề,Công ty,Tổng điểm (%),Điểm kỹ năng (%),Điểm văn bản (%),Lời giải thích,Mức lương
0,Data Analyst (Lead),Giải pháp Phân tích dữ liệu Insight Data,93.3,80.0,100.0,"Khớp 4/4 kỹ năng (SQL, Power BI, Excel); Kinh ...",Thoả thuận
1,Trưởng Nhóm Data Analyst,HỆ THỐNG Y TẾ MEVIS HEALTHCARE,83.1,83.3,85.3,"Khớp 5/5 kỹ năng (Python, SQL, Power BI); Yêu ...",20 - 30 tr
2,Data Analyst (Workforce Management Analyst)_Ec...,TRANSCOSMOS,83.0,75.0,76.1,"Khớp 3/3 kỹ năng (SQL, Power BI, Excel); Kinh ...",14 - 15 tr
3,Data Analyst Intern,GEMS UNITED JOINT STOCK COMPANY,82.8,68.1,85.3,"Khớp 3/4 kỹ năng (Power BI, Excel, Data Analys...",Thoả thuận
4,Chuyên Viên Phân Tích Dữ Liệu (Data Analyst) -...,Chi Nhánh Trung Tâm Công Nghệ Thông Tin Tập Đo...,78.8,80.0,67.2,"Khớp 4/4 kỹ năng (Power BI, Data Analysis, Tab...",Thoả thuận



 BÁO CÁO ĐÁNH GIÁ ĐỘ PHÙ HỢP HỒ SƠ VỚI CÔNG VIỆC (1:1 FIT REPORT)
Công việc mục tiêu: DevOps Engineer
Công ty:             Apero Technologies Group
Địa điểm:            Hà Nội (Onsite)
Mức lương:           20 - 45 tr

--- 1. TỔNG ĐIỂM ---
• Tổng điểm: 79.6%   |   Xếp hạng: 1/1547 tin còn hạn

--- 2. CÁC ĐIỂM THÀNH PHẦN (cùng công thức với recommend_jobs) ---
• Kỹ năng        : 54.0%   (khớp 9/15 kỹ năng, thêm 0 kỹ năng gần nhau tính 0,5)
• Văn bản        : 94.7%   (cosine gốc 46.9%, đã chuẩn hóa theo job giống nhất)
• Kinh nghiệm    : 100.0%   (Đạt yêu cầu (3.0 năm >= 2.0 năm))
• Vị trí/hình thức: 55.0%
• Lương          : 100.0%

--- 3. CHI TIẾT KỸ NĂNG ---
 Kỹ năng bạn đã có (9):
   Microservices, AWS, Docker, Kubernetes, CI/CD, Git, Linux, Jenkins, DevOps
 Kỹ năng gần với kỹ năng bạn có, học thêm nhanh (0):
   Không có
 Kỹ năng còn thiếu mà công ty yêu cầu (6):
   Apache Kafka, LLM, Redis, Nginx, Prometheus, Grafana
